Python 3.11 was used for this.

Originally made for hagrid dataset (30k sample, 512px due to size constraints) was downloaded and extracted externally.

Modified for HaGRIDv2 dataset.(900k dataset, 512px).

In [1]:
!python --version

Python 3.11.11


In [2]:
!pip install --upgrade pip
!pip install mediapipe-model-maker
!pip install seaborn matplotlib scikit-learn numpy

In [3]:
# Show directory structure of hagrid dataset
!ls -l HaGRIDv2_dataset_512

total 61680
drwxr-xr-x 2 nzoe nzoe 1961984 Sep  3  2024 call
drwxr-xr-x 2 nzoe nzoe 2199552 Aug 22  2024 dislike
drwxr-xr-x 2 nzoe nzoe 2174976 Aug 22  2024 fist
drwxr-xr-x 2 nzoe nzoe 2183168 Aug 22  2024 four
drwxr-xr-x 2 nzoe nzoe 2514944 Aug 22  2024 grabbing
drwxr-xr-x 2 nzoe nzoe 2514944 Aug 22  2024 grip
drwxr-xr-x 2 nzoe nzoe 2658304 Aug 22  2024 holy
drwxr-xr-x 2 nzoe nzoe 2088960 Aug 22  2024 like
drwxr-xr-x 2 nzoe nzoe 2498560 Aug 22  2024 little_finger
drwxr-xr-x 2 nzoe nzoe 2617344 Aug 22  2024 middle_finger
drwxr-xr-x 2 nzoe nzoe 2248704 Aug 22  2024 mute
drwxr-xr-x 2 nzoe nzoe  159744 May 21 00:45 no_gesture
drwxr-xr-x 2 nzoe nzoe 2162688 Aug 22  2024 ok
drwxr-xr-x 2 nzoe nzoe 2244608 Aug 22  2024 one
drwxr-xr-x 2 nzoe nzoe 2232320 Aug 22  2024 palm
drwxr-xr-x 2 nzoe nzoe 2248704 Aug 22  2024 peace
drwxr-xr-x 2 nzoe nzoe 2068480 Aug 22  2024 peace_inverted
drwxr-xr-x 2 nzoe nzoe 2600960 Aug 22  2024 point
drwxr-xr-x 2 nzoe nzoe 2224128 Aug 22  2024 rock
drwxr-xr-x 2 nzoe

In [4]:
import os
import shutil

# ORIGINAL_DATA_DIR = "hagrid-sample-500k-384p/hagrid_500k"
# ORIGINAL_DATA_DIR = "hagrid-sample-30k-384p/hagrid_30k"
ORIGINAL_DATA_DIR = "HaGRIDv2_dataset_512"
FORMATTED_DATA_DIR = "mediapipe_dataset"

if os.path.exists(FORMATTED_DATA_DIR):
    print(f"Removing existing dataset directory: {FORMATTED_DATA_DIR}")
    shutil.rmtree(FORMATTED_DATA_DIR)

os.makedirs(FORMATTED_DATA_DIR, exist_ok=True)

print("mapping dataset folders for MediaPipe...")
for folder_name in os.listdir(ORIGINAL_DATA_DIR):
    original_path = os.path.join(ORIGINAL_DATA_DIR, folder_name)
    
    # Ensure it's a directory and skip hidden files
    if os.path.isdir(original_path) and not folder_name.startswith('.'):
        
        # Map 'no_gesture' to 'none' for MediaPipe's background class requirement
        if folder_name == "no_gesture":
            class_name = "none"
        else:
            class_name = folder_name
        
        new_path = os.path.join(FORMATTED_DATA_DIR, class_name)
        
        # Use symlinks to link photos to each folder
        if not os.path.exists(new_path):
            try:
                os.symlink(os.path.abspath(original_path), os.path.abspath(new_path))
                print(f"Linked: {folder_name} -> {class_name}/")
            except OSError as e:
                print(f"Error linking {class_name}: {e}")

    # # This section was originally meant for a different directory structure.
    # # Irrelevant for HaGRIDv2's dataset.
    # if folder_name.startswith("train_val_"):
    #     # extract the class name (folders are named train_val_*)
    #     class_name = folder_name.replace("train_val_", "")
        
    #     original_path = os.path.join(ORIGINAL_DATA_DIR, folder_name)
    #     new_path = os.path.join(FORMATTED_DATA_DIR, class_name)
        
    #     # Use symlinks to link photos to each folder
    #     # I don't have the space to copy gigabytes of photos
    #     if not os.path.exists(new_path):
    #         try:
    #             os.symlink(os.path.abspath(original_path), os.path.abspath(new_path))
    #             print(f"Linked: {folder_name} -> {class_name}/")
    #         except OSError as e:
    #             print(f"Error linking {class_name}.")

print(f"\nDataset ready! Clean labels found: {os.listdir(FORMATTED_DATA_DIR)}")

Removing existing dataset directory: mediapipe_dataset
mapping dataset folders for MediaPipe...
Linked: three -> three/
Linked: point -> point/
Linked: peace_inverted -> peace_inverted/
Linked: three_gun -> three_gun/
Linked: holy -> holy/
Linked: peace -> peace/
Linked: two_up_inverted -> two_up_inverted/
Linked: middle_finger -> middle_finger/
Linked: grip -> grip/
Linked: stop_inverted -> stop_inverted/
Linked: fist -> fist/
Linked: three3 -> three3/
Linked: grabbing -> grabbing/
Linked: dislike -> dislike/
Linked: palm -> palm/
Linked: one -> one/
Linked: four -> four/
Linked: no_gesture -> none/
Linked: call -> call/
Linked: ok -> ok/
Linked: two_up -> two_up/
Linked: stop -> stop/
Linked: mute -> mute/
Linked: like -> like/
Linked: little_finger -> little_finger/
Linked: rock -> rock/
Linked: thumb_index -> thumb_index/
Linked: three2 -> three2/

Dataset ready! Clean labels found: ['three', 'point', 'peace_inverted', 'three_gun', 'holy', 'peace', 'two_up_inverted', 'middle_finger

In [5]:
import os

DATASET_PATH = "mediapipe_dataset"
none_dir = os.path.join(DATASET_PATH, "none")

# # Create the 'none' directory, needed by mediapipe_model_maker
# os.makedirs(none_dir, exist_ok=True)
# print(f"Created required background class directory: {none_dir}")

# Checks to ensure resources will not be wasted before training.

# Check if directory exists
assert os.path.exists(none_dir), "ERROR: The 'none' directory is completely missing. Stopping to prevent training failure."

# Check if directory actually contains images
img_count = len([f for f in os.listdir(none_dir) if not f.startswith('.')])
assert img_count > 0, "ERROR: The 'none' directory exists but contains 0 images (there should be some). Stopping to prevent training failure."

# If everything passes, continue.
print(f"Successfully verified 'none' class layout.")
print(f"Number of background images ready for training: {img_count}")

Successfully verified 'none' class layout.
Number of background images ready for training: 2164


In [ ]:
import os
import tensorflow as tf
from datetime import datetime
assert tf.__version__.startswith('2')

from mediapipe_model_maker import gesture_recognizer

DATASET_PATH = "mediapipe_dataset"
EXPORT_DIR = f"exported_model_{datetime.now():%Y%m%d_%H%M%S}"
os.makedirs(EXPORT_DIR, exist_ok=True)
print(f"Using export directory: {EXPORT_DIR}")

print("Loading local dataset into MediaPipe Model Maker...")
print("Note: This may take a while as the hand-tracker scans the images...")

data = gesture_recognizer.Dataset.from_folder(
    dirname=DATASET_PATH,
    hparams=gesture_recognizer.HandDataPreprocessingParams()
)

# Split the dataset: 80% for training, 10% for validation, 10% for testing
train_data, rest_data = data.split(0.8)
validation_data, test_data = rest_data.split(0.5)

print(f"Training Data Size: {len(train_data)}")
print(f"Validation Data Size: {len(validation_data)}")
print(f"Testing Data Size: {len(test_data)}")

hparams = gesture_recognizer.HParams(
    export_dir=EXPORT_DIR,
    epochs=5,
    learning_rate=0.001,
	batch_size=256,
	lr_decay=0.99
)

options = gesture_recognizer.GestureRecognizerOptions(hparams=hparams)

print("Starting model training...")
model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options)

2026-05-21 01:15:33.596075: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-21 01:15:33.617609: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-21 01:15:33.617625: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-21 01:15:33.618556: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-21 01:15:33.623587: I tensorflow/core/platform/cpu_feature_guar

Using export directory: exported_model_20260521_011535
Loading local dataset into MediaPipe Model Maker...
Note: This may take a while as the hand-tracker scans the images...


In [ ]:
print("Evaluating model...")
loss, acc = model.evaluate(test_data, batch_size=1)
print(f"Test loss: {loss}, Test accuracy: {acc}")

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score, precision_score, recall_score
import matplotlib.pyplot as plt
import seaborn as sns

print("Computing predictions on test data...")
# Get all predictions and labels from test data
all_predictions = []
all_labels = []

for sample in test_data:
    # MediaPipe returns results with hand landmarks
    result = model.recognize(sample.image)
    prediction = result.gestures[0][0].category_name if result.gestures and result.gestures[0] else "unknown"
    all_predictions.append(prediction)
    all_labels.append(sample.label)

# Get unique classes
classes = sorted(list(set(all_labels)))
print(f"Classes: {classes}")

# Compute confusion matrix
cm = confusion_matrix(all_labels, all_predictions, labels=classes)
print("\nConfusion Matrix:")
print(cm)

# Compute metrics
precision = precision_score(all_labels, all_predictions, average='weighted', zero_division=0)
recall = recall_score(all_labels, all_predictions, average='weighted', zero_division=0)
f1 = f1_score(all_labels, all_predictions, average='weighted', zero_division=0)

print(f"\nWeighted Metrics:")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

print(f"\nDetailed Classification Report:")
print(classification_report(all_labels, all_predictions, labels=classes, zero_division=0))

# Visualize confusion matrix
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Gesture Recognition Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(f'{EXPORT_DIR}/confusion_matrix.png', dpi=100, bbox_inches='tight')
print(f"\nConfusion matrix visualization saved to {EXPORT_DIR}/confusion_matrix.png")
plt.show()

In [ ]:
print("Exporting model...")
model.export_model()
print(f"Model exported successfully to {EXPORT_DIR}/gesture_recognizer.task")